# <font style="font-family:roboto;color:#455e6c"> Generating datasets for Machine Learning Interatomic Potentials with the ASSYST method </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Poul, M., Huber, L. & Neugebauer, J. Automated Generation of Structure Datasets for Machine Learning Potentials and Alloys.](https://www.nature.com/articles/s41524-025-01669-4) (2025)

## <font style="font-family:roboto;color:#455e6c"> Background </font> 

*Automated Small SYmetric Structure Training* or ASSYST is a method to generate training data for machine learning potentials.
The key idea is to use small structures to automatically explore structurally and chemically diverse atomic environments and provide training data around the energetically most favorable ones.

### <font style="font-family:roboto;color:#455e6c"> Workflow Overview </font>

<div style="text-align: center;">
  <img src="img/AssystSchematicModified.png" alt="ASSYST workflow schematic" width="600"/>
</div>

### <font style="font-family:roboto;color:#455e6c"> Covering Potential Energy Surface </font>

<div style="text-align: center;">
  <img src="img/PES_surface_modified.png" alt="ASSYST workflow schematic" width="800"/>
</div>

Consider the simplified one-dimensional potential energy surface (PES) vs the configuration space with a global minimum and multiple local minima. Starting with the structures generated from spacegroup sampling (<span style="color: blue;">blue points</span>), they fall onto different locations on the PES curve. To cover the different configuration space, we can do a volume relaxation (<span style="color: orange;">orange points</span>) that covers points closer to the minima. Then, to get to the minima we can do a full relaxation with atoms and volume (<span style="color: green;">green points</span>). However, to cover the space of higher energy range, we can stretch the structures by applying different strains (e.g., shear, hydrostatic) which represents structural deformations or rattle the atomic positions in the structure (<span style="color: purple;">purple points</span>) which would cover the phonon properties.

By sampling the space even more, we improve the description of the potential energy surface, hence the transferability of the potentials trained on such datasets.

### <font style="font-family:roboto;color:#455e6c"> Transferability </font>

ASSYST trained potentials describe also structures that they are not directly trained on, such as point and planar defects.

<div style="text-align: center;">
  <img src="img/Fig8_MTP24_2d0_8d2_DefectsManual.png" alt="Defects transferability" width="800"/>
</div>

Liquid state is also well described and potentials are stable for long running thermodynamic integrations.

<div style="text-align: center;">
  <img src="img/Fig11_MgCa.png" alt="Mg–Ca phase behavior" width="800"/>
</div>

This phase diagram is our goal for today!

### <font style="font-family:roboto;color:#455e6c"> Literature </font>

- Mg and Defects: https://journals.aps.org/prb/abstract/10.1103/PhysRevB.107.104103
- Ternary Mg/Al/Ca: https://www.nature.com/articles/s41524-025-01669-4

## <font style="font-family:roboto;color:#455e6c"> Spacegroup Sampling </font> 

The first step in the ASSYST workflow is to decide which chemical space to cover and how densely. Increasing the new number of total atoms allows you to generate more and more complex structures and also sample the chemical space more densely.

This example is for 'Mg' unary system, where we sampled 1-10 Atoms in the structures.

<div style="text-align: center;">
  <img src="img/Mg_spacegroup.png" alt="Mg–Ca phase behavior" width="800"/>
</div>

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Build a small workflow that creates a dataframe table of spacegroup sampling Mg and Ca. Consider the following steps,
1. Load the `SpaceGroup` workflow.
2. In each structure, `Mg` and `Ca` should have a minimum of `2` atoms and a maximum of `4` atoms. (Check the output of `Multiply` node using `StoichiometryTable`)
3. Add `SpaceGroupSampling` node from node library and connect it with the output of `Multiply`.
4. Limit the maximum number of sampled structures to `10`. Each sampled structure should contain at most `8` atoms.
5. Plot the space group sampling.

**BONUS: Plot the energy Vs volume distribution.**

Check `atomistic` -> `assyst` for all the nodes.
</div>

In [1]:
from core import Workflow
from core.gui import PyironFlow, GUILayout

/cmmc/ptmp/pyironhb/mambaforge/envs/pyiron_mpie_cmti_2026-02-16/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
from pyiron_nodes.atomistic.assyst.plot import PlotSpaceGroups, PlotEnergyVolumeHistogram
from pyiron_nodes.atomistic.assyst.stoichiometry import SpaceGroupSampling, ElementInput, StoichiometryTable
from pyiron_nodes.basic.math import Multiply
from pyiron_nodes.atomistic.calculator.optimize import RelaxLoop, GenericOptimizerSettings
from pyiron_nodes.atomistic.structure.transform import RattleLoop, StretchLoop
from pyiron_nodes.atomistic.engine.grace import GRACE
from pyiron_nodes.basic.math import Multiply
from pyiron_nodes.atomistic.structure.group import CombineStructures

In [ ]:
def make_spg(
        name, *elements,
        min_atoms=2, max_atoms=4, max_structures=50,
        delete_existing_savefiles=False
):
    wf = Workflow(name)

    element_nodes = []
    if len(elements) > 0:
        e1, *elements = elements
        stoi = ElementInput(e1, min_atoms=min_atoms, max_atoms=max_atoms)
        setattr(wf, 'Element_1', stoi)
        element_nodes.append(stoi)
        for i, e in enumerate(elements):
            en = ElementInput(e, min_atoms=min_atoms, max_atoms=max_atoms)
            setattr(wf, f'Element_{i+2}', en)
            element_nodes.append(en)
            stoi = Multiply(stoi, en)
            setattr(wf, f'Multiply_{i+1}_{i+2}', stoi)
        # if len(elements) > 0:
        #     wf.Multiply = stoi
        spg = SpaceGroupSampling(
                elements=stoi,
                spacegroups=None,
                max_atoms=len(element_nodes) * max_atoms,
                max_structures=max_structures
        )
    else:
        spg = SpaceGroupSampling(
                max_atoms=max_atoms,
                max_structures=max_structures
        )
    plotspg = PlotSpaceGroups(spg)
    plothistogram = PlotEnergyVolumeHistogram(spg)
    stoichiometry =  StoichiometryTable(stoi)
    wf.SpaceGroupSampling = spg
    wf.StoichiometryTable = stoichiometry
    wf.PlotSPG = plotspg
    wf.PlotEnergyVolumeData = plothistogram
    return wf

In [ ]:
elements = ['Mg', 'Ca']
wf = make_spg('SPG_Solution', *elements, max_structures= 10)

In [ ]:
layout = GUILayout(flow_widget_width=800, flow_widget_height=600)
pf = PyironFlow([wf], gui_layout = layout,nodes_path="pyiron_nodes")
pf.gui

<div style="text-align: center;">
  <img src="img/spacegroup_example.png" alt="Mg–Ca phase behavior" width="100%"/>
</div>

## <font style="font-family:roboto;color:#455e6c"> Full Workflow for a Small Structure Set </font> 

This demonstration uses the GRACE universal force fields for the relaxation steps. Usually, we would run them in low-convergence DFT and then calculate each structure with high-convergence DFT.

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Build a complete ASSYST workflow that creates a complete dataframe table of spacegroup sampling Mg and Ca. Consider the following steps,
1. Load the `ASSYST` workflow.
2. Add `StretchLoop` and `RattleLoop` nodes (Note: `Stretch` and `Rattle` nodes only work on a single structure and not a dataframe of structures).
3. Add `CombineStructures` node to combine all the dataframes that we get.
4. Finally, plot the energy Vs volume distribution.

**Hint:** Check the workflow diagram to see how can you connect the new nodes.

**Optional:** you can tick the option to save the dataframe into a file by providing a path or a name of `filename`.

Check `atomistic` -> `structure` for relevant structure nodes and `atomistic` -> `engine` for `GRACE` engine.
</div>

In [ ]:
def make_assyst(
        name, *elements,
        min_atoms=2, max_atoms=4, max_structures=50,
        delete_existing_savefiles=False
):
    wf = Workflow(name)
    
    element_nodes = []
    engine = GRACE()
    wf.GraceEngine = engine
    if len(elements) > 0:
        e1, *elements = elements
        stoi = ElementInput(e1, min_atoms= min_atoms, max_atoms= max_atoms)
        setattr(wf, 'Element_1', stoi)
        element_nodes.append(stoi)
        for i, e in enumerate(elements):
            en = ElementInput(e, min_atoms= min_atoms, max_atoms= max_atoms)
            setattr(wf, f'Element_{i+2}', en)
            element_nodes.append(en)
            stoi = Multiply(stoi, en)
            setattr(wf, f'Multiply_{i+1}_{i+2}', stoi)
        spg = SpaceGroupSampling(
                elements= stoi,
                spacegroups= None,
                max_atoms= len(element_nodes) * max_atoms,
                max_structures= max_structures, engine= engine
        )
    else:
        spg = SpaceGroupSampling(
                max_atoms= max_atoms,
                max_structures= max_structures,
                engine= engine
        )
    
    plotspg = PlotSpaceGroups(spg)
    
    optimizer_settings = GenericOptimizerSettings()

    volume_relax = RelaxLoop(mode="volume", 
                             engine=engine,
                             opt=optimizer_settings, df_structures=spg)
    full_relax = RelaxLoop(mode="full", 
                           engine=engine,
                           opt=optimizer_settings,
                           df_structures=volume_relax)

    rattle = RattleLoop(
            df_structures=full_relax,
            sigma=0.25,
            samples=4,
            engine= engine
    )

    stretch = StretchLoop(
            df_structures=full_relax,
            hydro=0.8, # 0.8
            shear=0.2, # 0.2
            samples=4,
            engine= engine
    )

    combine_structures = CombineStructures(
            spacegroups= spg.outputs.df_structures,
            volume_relax= volume_relax.outputs.df_relaxed,
            full_relax= full_relax.outputs.df_relaxed,
            rattle= rattle.outputs.df_rattled,
            stretch= stretch.outputs.df_stretched,
            filename = "data/Structures_Everything",
            store = False
    )
    
    plothistogram = PlotEnergyVolumeHistogram(combine_structures)

    wf.SpaceGroupSampling = spg
    wf.PlotSPG = plotspg
    wf.OptimizerSettings = optimizer_settings
    wf.VolumeRelax = volume_relax
    wf.FullRelax = full_relax

    wf.Rattle = rattle
    wf.Stretch = stretch

    wf.CombineStructures = combine_structures
    wf.PlotEnergyVolumeData = plothistogram
    return wf

In [ ]:
elements = ['Mg', 'Ca']
wf = make_assyst('ASSYST_solution', *elements, max_structures= 10)

In [ ]:
layout = GUILayout(flow_widget_width=800, flow_widget_height=600)
pf = PyironFlow([wf], gui_layout = layout,nodes_path="pyiron_nodes")
pf.gui

<div style="text-align: center;">
  <img src="img/assyst_example.png" alt="Mg–Ca phase behavior" width="100%"/>
</div>

## <font style="font-family:roboto;color:#455e6c"> Precomputed Full Workflow with Large Structure Set </font> 

This is the complete unary `Mg` dataset if we allowed for high-convergence DFT without tight constraints on structure generation which contains ~9k structures.

<div style="text-align: center;">
  <img src="img/mg_dataset.png" alt="" width="100%"/>
</div>

<img src="img/logo_roll.png" width="1200">